<a href="https://colab.research.google.com/github/IAmTheInfinite/github-slideshow/blob/main/IntegracionNumerica_MC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> (Última Actualización: 4 de Junio de 2026)

# Introducción a la Mecánica Computacional
## Mecánica Clásica - Cátedra G. Mindlin - 1er Cuatrimestre 2026

En el marco de la materia *Mecánica Clásica*, de la Licenciatura en Ciencias Físicas, vamos a ver métodos numéricos para encontrar las soluciones a las ecuaciones diferenciales asociadas a ecuaciones de movimiento de manera general, y su relación y aplicación a problemas específicos, de manera tal de complementar los contenidos vistos en la teórica y en la práctica.

# Motivación de este colab
<!-- $$\require{amsmath}$$ -->

En la mecánica clásica, los distintos formalismos nos proveen de herramientas teóricas robustas para deducir las leyes que gobiernan la evolución temporal de un sistema. Hemos explorado distintos formalismos analíticos que nos permitieron llegar a las **ecuaciones de movimiento** y en algunos casos encontrar las soluciones explícitas a partir de su integración. Pero para un número importante de los sistemas de interés físico real, estas ecuaciones no poseen soluciones analíticas cerradas. Aún así, para estos problemas podemos analíticamente avanzar en la comprensión de un sistema a partir de: encontrar soluciones explícitas a versiones linealizadas (locales) de las ecuaciones de movimiento; reducir dimensiones del problema aprovechando simetrías y leyes de conservación (Teorema de Noether); identificar relaciones entre soluciones (como determinar la geometría de una órbita elíptica); realizar un estudio cualitativo del movimiento a través del potencial efectivo y la energía, y/o del análisis de puntos de equilibrio y su estabilidad ante perturbaciones. Sin embargo, puede resultarnos de interés el encontrar las soluciones del sistema aún cuando las ecuaciones no pueden integrarse analíticamente (o no sea fácil hacerlo). En estos casos, podemos encontrar soluciones numéricas dentro de lo que se podría denominar como **Mecánica Numérica o Computacional**.


---

Ecuaciones de Movimiento según el Formalismo

| **Mecánica Newtoniana** | **Mecánica Lagrangiana** | **Mecánica Hamiltoniana** |
| :---: | :---: | :---: |
| Basado en fuerzas empíricas y coordenadas cartesianas. | Basado en el Principio de Hamilton y ecuaciones de ligadura implícitas. | Basado en el espacio de fases y la geometría simpléctica. |
| $$\mathbf{F} = m\mathbf{a}$$|$$\frac{d}{dt}\left(\frac{\partial L}{\partial \dot{q}_i}\right) - \frac{\partial L}{\partial q_i} = 0$$|$$\dot{q}_i = \frac{\partial H}{\partial p_i}, \quad \dot{p}_i = -\frac{\partial H}{\partial q_i}$$ |

---

Para resolver estos problemas de manera computacional, planteamos las ecuaciones como un problema de valor inicial estructurado en un sistema de ecuaciones diferenciales ordinarias (ODEs) de primer orden en un espacio $2n$-dimensional:

$$\frac{d\vec{x}}{dt}=\vec{f}(t,\vec{x})$$

Mientras que los métodos de Newton o Lagrange nos proveen de ecuaciones de segundo orden, la formulación hamiltoniana realiza de forma nativa y elegante la reducción de orden a un sistema de primer orden en $2n$ variables. Esta estructura es el punto clave para la computación, ya que permite que los integradores numéricos estándar (como los métodos de Euler, Runge-Kutta de orden 4 o las librerías modernas como `scipy.integrate.solve_ivp`) resuelvan eficientemente el sistema para devolvernos las trayectorias temporales y el mapa del diagrama de fases.


In [ ]:
# # # Importamos paquetes
# %matplotlib inline
import numpy as np
np.set_printoptions(legacy='1.25')
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
# from matplotlib.gridspec import GridSpec
# from matplotlib.colors import Normalize
# import matplotlib.cm as cm
# import warnings
# warnings.filterwarnings('ignore')
# # Estilo global de figuras
# plt.rcParams.update({
#     'figure.dpi': 120,
#     'font.size': 12,
#     'axes.labelsize': 13,
#     'axes.titlesize': 14,
#     'lines.linewidth': 1.5,
#     'legend.fontsize': 11,
# })

---
# Métodos de integración numérica

### [I. Método de Euler](#euler)
### [II. Método de Runge-Kutta](#rk4)
### [III. Funciones integradas en Scipy](#scipy)

---
<a name="euler"></a>
## I. Método de Euler

Supongamos que tenemos una ecuación diferencial de esta forma:

$$
\frac{dx}{dt} = f(t, x)
$$

El método de Euler consiste en aproximar la derivada por el cociente incremental: $\frac{dx}{dt} \simeq \frac{\Delta x}{\Delta t}$:

$$
\frac{x(t+\Delta t) - x(t)}{\Delta t} = f(t, x)
$$

$$
x(t+\Delta t) = x(t)+f(t, x)\Delta t
$$

Esto define una regla que prescribe una aproximación para el valor de $x$, después de cierto tiempo, en base a la condición inicial y el valor de la derivada (campo vector).

Podemos definir un paso temporal discreto $h$, por lo que nos queda una regla iterativa para integrar el $x$ de a pasos discretos

$${x}_{n+1} = {x}_{n} + h f(t_{n},{x}_{n})$$

Notar que es como aproximar a la variable por su desarrollo de Taylor de orden 1, donde me muevo un paso $h$ para donde me dice la derivada primera (que es el campo vector por la ODE). Necesito una condición inicial.

**Qué pasa si tengo más dimensiones?**

Supongamos que tenemos una sistema de ecuación diferenciales del tipo:

$$\dot{x} = f(t, x, y)$$
$$\dot{y} = g(t, x, y)$$

Con una condición inicial dada $({x}_{0},{y}_{0})$.

Como vimos, la integración numérica consiste en aproximar las soluciones de un conjunto de ecuaciones diferenciales utilizando una discretización en la variable temporal. En el caso del método de Euler, vamos a aproximar mediante un mapa que va de $({x}_{n},{y}_{n})$ a $({x}_{n+1},{x}_{n+1})$.

Luego, un paso de integración mediante el método de Euler 2D será:

$${x}_{n+1} = {x}_{n} + h f(t_{n}, {x}_{n}, {y}_{n})$$
$${y}_{n+1} = {y}_{n} + h g(t_{n}, {x}_{n}, {y}_{n})$$

Esto también se puede expresar de manera vectorial, con posibilidad de generalizarse a más dimensiones

$${\vec{x}}_{n+1} = {\vec{x}}_{n} + h \vec{f}(t_{n}, {\vec{x}}_{n})$$

Que resuelve el sistema de ecuaciones general escrito de manera vectorial

$$\frac{d\vec{x}}{dt} = \vec{f}(t, \vec{x})$$


---
### **Ejemplo**

Veamos una aplicación como ejemplo. Supongamos la ODE

$$
\dot{N} = -aNln(bN)
$$

Entonces,

$$
\frac{N(t + \Delta t) - N(t)}{\Delta t} = -aN(t) ln(bN(t))
$$

$$
N(t + \Delta t) = N(t) -aN(t) ln(bN(t))\Delta t
$$

Para resolver esto tenemos que definir la condición incial y el paso temporal. Qué les parece que puede pasar si cambiamos sus valores?

In [ ]:
# Definimos el valor de los parámetros del modelo
a = 0.5
b = 1
# Damos el paso de integración o evolución
dt = 0.1
# Damos el vector de los tiempos para los que vamos a calcular N
t = np.arange(0, 10, step=dt)
# Nos armamos un vector N donde vamos a ir guardando los resultados de la integración. Por ahora son todos ceros. Lo vamos llenando al ir evolucionando con Euler.
N = np.zeros_like(t)
# Fijamos una condición inicial de N, dandole valor al primer punto
N[0] = 3
#Y ahora sí ya hacemos la evolución de la formulita
for i in range(len(t)-1):
    # Calculamos el siguiente punto usando el método de Euler (t -> i ; t+dt -> i+1)
    N[i + 1] = -a*N[i]*np.log(b*N[i]) * dt + N[i]

In [ ]:
# Graficamos
plt.figure(figsize=(5,5))
plt.plot(t, N, 'k.-', lw=2, markersize=10)
plt.scatter(t[0], N[0], color="red", s=50, label="c.i", zorder=3) # Hacemos el puntito de c.inicial
plt.xlabel("Tiempo (u.a)", fontsize=16)
plt.ylabel("N(t)", fontsize=16, rotation=0, labelpad=20)
plt.legend(fontsize=14)
plt.show()

**Paso temporal**

En general, cuando uso el método de integración de Euler, tengo que calibrar con cuidado el paso temporal, para asegurarme de no estar haciendo cualquier cosa. Una forma de hacer esto es hacer la integración para pasos temporales sucesivamente menores, hasta notar (con algún criterio) que la integración no cambia sustancialmente al refinar el paso temporal.

In [ ]:
varios_dt = [1, 0.3, 0.1, 0.03, 0.01]
plt.figure(figsize=(5,5))
for dt in varios_dt:
    # Para cada paso de integración hacemos el proceso
    t = np.arange(0, 10, step=dt)
    a = 0.5
    b = 1
    N = np.zeros_like(t)
    # Fijamos una condición inicial, dandole valor al primer punto
    N[0] = 3
    for i in range(len(t)-1):
        # Calculamos el siguiente punto usando el método de Euler (t -> i ; t+dt -> i+1)
        N[i + 1] = -a*N[i]*np.log(b*N[i]) * dt + N[i]
    # Le agregamos una etiqueta a cada curva para saber que dt le correspondía
    plt.plot(t, N, label='dt = {}'.format(dt))
plt.xlabel("Tiempo (u.a)", fontsize=16)
plt.ylabel("N(t)", fontsize=16)
plt.legend()
plt.show()

**Condiciones iniciales**

Integremos ahora para varias condiciones iniciales.

In [ ]:
# Damos el valor de parametros y paso de integracion
a = 0.5
b = 1
dt = 0.1
# Damos el vector de los tiempos para los que vamos a calcular N
t = np.arange(0, 10, step=dt)
# Definimos el vector N1, por ahora son todos ceros. Lo vamos llenando al ir evolucionando con Euler.
N1 = np.zeros_like(t)
# Damos la condicion inicial de N1.
N1[0] = 3
# Y ahora sí ya hacemos la evolución de la formulita.
for i in range(len(t)-1):
    N1[i + 1] = -a*N1[i]*np.log(b*N1[i]) * dt + N1[i]

# Ahora integramos para otra condicion inicial
N2 = np.zeros_like(t)
# Damos otra condicion inicial
N2[0] = 0.1
# La integramos
for i in range(len(t)-1):
    N2[i + 1] = -a*N2[i]*np.log(b*N2[i]) * dt + N2[i]

#Y ahora mostramos las dos
plt.figure(figsize=(5,5))
plt.plot(t, N1)
plt.plot(t, N2)
plt.xlabel("Tiempo (u.a)", fontsize=16)
plt.ylabel("N(t)", fontsize=16)
plt.show()

In [ ]:
# Y también podríamos hacer un loop y que nos integre muchas condiciones iniciales
condiciones_iniciales = np.arange(0.1, 3, 0.1)
plt.figure(figsize=(5,5))

for ci in condiciones_iniciales:
    N = np.zeros_like(t)
    # Damos la condicion inicial de N.
    N[0] = ci
    # Y ahora sí ya hacemos la evolución de la formulita.
    for i in range(len(t)-1):
        N[i + 1] = -a*N[i]*np.log(b*N[i]) * dt + N[i]
    plt.plot(t, N)
plt.xlabel("Tiempo (u.a)", fontsize=16)
plt.ylabel("N(t)", fontsize=16)
plt.show()

---
### Ejercicio 1

Escriba una función llamada `odeEuler` que tome como argumentos de entrada el campo vector `f(x)`, la condición inicial `ci`, el paso de integración `h` seteado por default como 0.1, y el tiempo total de integración `tmax` seteado por default como 10. La función debe devolver el vector de tiempos y los valores de la variable en cada paso.

In [ ]:
# # # COMPLETAR

In [ ]:
# # # SOLUCION
def odeEuler(f, ci, h=0.1, tmax=10):
    t_grid = np.arange(0,tmax,h)
    x = np.zeros(len(t_grid))
    x[0] = ci
    for i in range(len(t_grid)-1):
        x[i+1] = x[i] + h*f(x[i])
    return t_grid, x

Pruebe la función para integrar la ODE del ejemplo anterior para los parámetros y condiciones iniciales previamente usados (defina una función `gompertz` con el modelo de antes para ingresar al integrador).

In [ ]:
# # # COMPLETAR

In [ ]:
# # # SOLUCION
def gompertz(x):
    a = 0.5
    b = 1
    return -a*x*np.log(b*x)

t_grid, x = odeEuler(gompertz, 3, 0.1, 10)
plt.figure(figsize=(5,5))
plt.plot(t_grid, x)
plt.xlabel("Tiempo (u.a)", fontsize=16)
plt.ylabel("N(t)", fontsize=16)
plt.show()

---
<a name="rk4"></a>
## II. Método de [Runge Kutta](https://en.wikipedia.org/wiki/Runge%E2%80%93Kutta_methods)

Así como podía pensar el método de Euler como una aproximación con Taylor de orden 1, podría considerar generalizar el método para ordenes superiores. Pero esto requiere el cálculo de derivadas, lo cual es un problema. Una alternativa es el método de punto medio que consiste básicamente en usar Euler, pero usando la información de la derviada en el punto medio de un paso de Euler. Este método es de orden 2 y pertence a una familia llamada métodos de **Runge-Kutta**.

$${x}_{n+1} = {x}_{n} + h f(t_n + \frac{h}{2}, x_n + \frac{h}{2} f(t_{n},{x}_{n}))$$

El método de Runge-Kutta de orden 4 (RK4) sigue la misma lógica, definiendo 4 evaluaciones intermedias:

$$k_1 = f(t_n, x_n)\\
k_2 = f\left(t_n + \frac{h}{2}, x_n + \frac{h}{2} k_1\right)\\
k_3 = f\left(t_n + \frac{h}{2}, x_n + \frac{h}{2} k_2\right)\\
k_4 = f(t_n + h, x_n + h k_3)$$

La actualización de $x$ se realiza mediante:

$$x_{n+1} = x_n + \frac{h}{6} (k_1 + 2k_2 + 2k_3 + k_4)$$

RK4 es el caballito de batalla de los integradores numéricos, pero, como todo, tiene sus ventajas y desventajas, y va a funcionar mejor o peor según el sistema.

Escribimos una función para usar este método. Esta función ejecuta la integración de **un paso temporal**. Los argumentos que requiere son: i) campo vector (**función**) ii) valor de las variables en el tiempo t, iii) paso temporal.

Los últimos dos (\*args, **kwargs) son para que, en caso de que sus campos vectores tengan argumentos, se los puedan pasar a la función y los sepa manejar.

In [ ]:
def rk4(dxdt, x, t, dt, *args, **kwargs):
    x = np.asarray(x)
    k1 = np.asarray(dxdt(x, t, *args, **kwargs))*dt
    k2 = np.asarray(dxdt(x + k1*0.5, t, *args, **kwargs))*dt
    k3 = np.asarray(dxdt(x + k2*0.5, t, *args, **kwargs))*dt
    k4 = np.asarray(dxdt(x + k3, t, *args, **kwargs))*dt
    return x + (k1 + 2*k2 + 2*k3 + k4)/6

Veamoslo con el ejemplo de antes ("gompertz"). Primero definimos el campo vector. Este tiene que ser una **función** que devuelva la derivada en el punto (o una lista con las derivadas de cada componente).

In [ ]:
def dNdt(N, t):
    return -N*np.log(N)

In [ ]:
# Definimos el paso temporal y el vector de tiempos
dt = 0.5
t = np.arange(0, 10, step=dt)
# Damos la condición inicial
N0 = 3
# Nos creamos un vector para guardar la solución
Nrk = np.zeros_like(t)
Nrk[0] = N0
for ix, tt in enumerate(t[:-1]):
    # Avanzamos un paso temporal
    Nrk[ix+1] = rk4(dNdt, Nrk[ix], tt, dt)
plt.figure(figsize=(5,5))
plt.plot(t, Nrk, 'o-')
plt.xlabel("Tiempo (u.a)", fontsize=16)
plt.ylabel("N(t)", fontsize=16)
plt.show()

De nuevo, es una buena práctica chequear que al achicar el paso de integración no cambie el resultado

In [ ]:
# Definimos el paso temporal y el vector de tiempos
dt = 0.5
t = np.arange(0, 10, step=dt)
# Damos la condición inicial
N0 = 3
# Nos creamos un vector para guardar la solución
Nrk = np.zeros_like(t)
Nrk[0] = N0
for ix, tt in enumerate(t[:-1]):
    # Avanzamos un paso temporal
    Nrk[ix+1] = rk4(dNdt, Nrk[ix], tt, dt)
plt.plot(t, Nrk, 'o-', label='dt = {:.2f}'.format(dt))

dt2 = 0.25
t2 = np.arange(0, 10, step=dt2)
# Damos la condición inicial
N0 = 3
# Nos creamos un vector para guardar la solución
Nrk = np.zeros_like(t2)
Nrk[0] = N0
for ix, tt in enumerate(t2[:-1]):
    # Avanzamos un paso temporal
    Nrk[ix+1] = rk4(dNdt, Nrk[ix], tt, dt2)

plt.plot(t2, Nrk, 'o-', label='dt = {:.2f}'.format(dt2))
plt.xlabel("Tiempo (u.a)", fontsize=16)
plt.ylabel("N(t)", fontsize=16)
plt.legend()
plt.show()

Con parámetros

In [ ]:
def rk4(dxdt, x, t, dt, *args, **kwargs):
    x = np.asarray(x)
    k1 = np.asarray(dxdt(x, t, *args, **kwargs))*dt
    k2 = np.asarray(dxdt(x + k1*0.5, t, *args, **kwargs))*dt
    k3 = np.asarray(dxdt(x + k2*0.5, t, *args, **kwargs))*dt
    k4 = np.asarray(dxdt(x + k3, t, *args, **kwargs))*dt
    return x + (k1 + 2*k2 + 2*k3 + k4)/6

In [ ]:
def dNdt_ab(N, t, a, b):
    return -a*N*np.log(b*N)

In [ ]:
a = 0.5
b = 1
dt = 0.5
t = np.arange(0, 10, step=dt)
# Damos la condición inicial
N0 = 3
# Nos creamos un vector para guardar la solución
Nrk = np.zeros_like(t)
Nrk[0] = N0
for ix, tt in enumerate(t[:-1]):
    # Avanzamos un paso temporal
    Nrk[ix+1] = rk4(dNdt_ab, Nrk[ix], tt, dt, a, b)
plt.plot(t, Nrk, 'o-', label='dt = {:.2f}'.format(dt))
plt.xlabel("Tiempo (u.a)", fontsize=16)
plt.ylabel("N(t)", fontsize=16)
plt.show()

---
### Ejercicio 2

Muestre los resultados de integrar el problema anterior para distintos valores del parámetro b.

In [ ]:
# # # COMPLETAR

In [ ]:
# # # SOLUCION
a = 0.5
bs = [0.01, 0.1, 1, 10]
dt = 0.5
t = np.arange(0, 10, step=dt)
# Damos la condición inicial
N0 = 3
for b in bs:
    # Nos creamos un vector para guardar la solución
    Nrk = np.zeros_like(t)
    Nrk[0] = N0
    for ix, tt in enumerate(t[:-1]):
        # Avanzamos un paso temporal
        Nrk[ix+1] = rk4(dNdt_ab, Nrk[ix], tt, dt, a, b)
    plt.plot(t, Nrk, 'o', label='b = '+str(b))
plt.xlabel("Tiempo (u.a)", fontsize=16)
plt.ylabel("N(t)", fontsize=16)
plt.legend()
plt.show()

---
La generalización de RK4 a ordenes superiores es directa, pero en lugar de considerar a las variables como escalares, ahora son vectores

$$\vec{K}_1 = \vec{f}(t_n, \vec{x}_n)\\
\vec{K}_2 = \vec{f}\left(t_n + \frac{h}{2}, \vec{x}_n + \frac{h}{2} \vec{K}_1\right)\\
\vec{K}_3 = \vec{f}\left(t_n + \frac{h}{2}, \vec{x}_n + \frac{h}{2} \vec{K}_2\right)\\
\vec{K}_4 = \vec{f}(t_n + h, \vec{x}_n + h \vec{K}_3)$$

La actualización de $\vec{x}$ se realiza mediante:

$$\vec{x}_{n+1} = \vec{x}_n + \frac{h}{6} (\vec{K}_1 + 2\vec{K}_2 + 2\vec{K}_3 + \vec{K}_4)$$

La forma de utilizarlo sería:

`x[i+1], y[i+1] = rk4(campo_vector, [x[i], y[i]], tt, dt)`

Lo aplicamos por ejemplo al sistema

$$
\dot{x} = y
$$
$$
\dot{y} = -x
$$


In [ ]:
# # # Ahora vamos a definir una función "campo_vector" con el sistema de ecuaciones, en formato compatible con nuestra función rk4
def campo_vector(z, t):
    # Como ahora las variables vienen en una lista (en el primer argumento: z)
    # primero las separamos para que sea más claro
    x = z[0]
    y = z[1]
    # Y ahora calculamos las derivadas
    dxdt = y
    dydt = -x
    return [dxdt, dydt]

# Integración Runge-Kutta 4
dt = 0.01
tmax = 10
t = np.arange(0, tmax, step=dt)
xrk = np.zeros_like(t)
yrk = np.zeros_like(t)
xrk[0] = 4
yrk[0] = 0
for ix, tt in enumerate(t[:-1]):
    xrk[ix+1], yrk[ix+1] = rk4(campo_vector, [xrk[ix], yrk[ix]], tt, dt)
plt.figure(figsize=(5,5))
plt.plot(xrk, yrk)
plt.plot(xrk[0], yrk[0], 'ko')
plt.xlabel('x', fontsize=12)
plt.ylabel('y', fontsize=12)
plt.show()

---
# Error de truncamiento local

Sea $y_1$ la aproximación de $y(t_1)$ mediante un paso de algún método numérico que utilice un tamaño de paso $h = t_1 - t_0$. El **error de truncamiento (local)** (para la ecuación diferencial y el método dados) es

$$
E(h) = | y(t_1) - y_1 |
$$

La palabra *local* significa que estamos viendo solo un paso del método y la palabra *truncamiento* tiene que ver con el truncamiento de la serie de Taylor.

La mayoría de los métodos numéricos se basan en la serie de Taylor, por lo que el error puede expresarse en términos del teorema de Taylor. Por ejemplo, considere la serie de Taylor hasta el orden $p$ evaluada en $t_1 = t_0 + h$:

$$
y(t_1) = y(t_0) + y'(t_0)h + \cdots + \frac{y^{(p)}(t_0)}{p!} h^p + \frac{y^{(p+1)}(c)}{(p+1)!} h^{p+1}
$$

para algún $c \in [t_0,t_1]$. Si un método numérico calcula $y(t_1)$ utilizando el polinomio de Taylor de grado $p$, entonces el error de truncamiento local es

$$
E(h) = | y(t_1) - y_1 | = \frac{| y^{(p+1)}(c) |}{(p+1)!} h^{p+1}
$$

Por lo tanto, podemos decir aproximadamente que un método numérico es de orden $p$ si el error de truncamiento local se parece a $Ch^{p+1}$ para alguna constante $C$.

Más precisamente, un método numérico es de **orden** $p$ si el error de truncamiento local satisface

$$
E(h) \leq C h^{p+1}
$$

para *cualquier* ecuación $y' = f(t,y)$, $y(t_0)=y_0$. La constante $C$ depende de $f$. Nótese que el orden es un entero positivo.

Generalmente es bastante difícil determinar el orden de un método numérico dada la fórmula. En cambio, podemos determinar el orden experimentalmente. La idea es que el error de truncamiento local debe satisfacer

$$
E(h) \approx C h^{p+1}
$$

cuando se aplica a la mayoría de las ecuaciones diferenciales. Por lo tanto, podemos observar la pendiente en el gráfico log-log:

$$
\log(E(h)) \approx (p+1) \log(h) + \log(C)
$$

El procedimiento para determinar experimentalmente el orden $p$ de un método numérico es:

1. Aplicar el método numérico a la ecuación $y' = y,y(0)=1$ para diferentes tamaños de paso $h_1$ y $h_2$.
2. Calcular los errores de truncamiento locales $E(h_1)$ y $E(h_2)$ utilizando la solución exacta $y(t)=e^t$.
3. Calcule la pendiente del gráfico logarítmico:

$$
p+1 \approx \frac{\log(E(h_2)) - \log(E(h_1))}{\log(h_2) - \log(h_1)}
$$

## Orden de los métodos

El método de Euler se construye utilizando el polinomio de Taylor de grado 1. El teorema de Talyor dice:

$$
y(t_1) = y(t_0) + y'(t_0)(t_1 - t_0) + \frac{y''(c)}{2}(t_1 - t_0)^2
$$

para algún $c \in [t_0,t_1]$. Por lo tanto, si $|y''(t)|\leq K_2$ para todo $t \in [t_0,t_1]$, entonces

$$
E(h) = \left| \frac{y''(c)}{2}(t_1 - t_0)^2 \right| \leq \frac{K_2 h^2}{2}
$$

Por lo tanto, el método de Euler es de orden 1. Verifiquemos experimentalmente el orden del método de Euler trazando el error de truncamiento local para el método de Euler aplicado a $y'=y$, $y(0)=1$.

In [ ]:
f = lambda y: y
y0 = 1;
h = [0.1,0.05,0.01,0.005,0.001]
E = np.zeros(len(h))
for n in range(0,len(h)):
    t_grid, y = odeEuler(f, y0, h[n])
    y1 = y[1]
    y1_exact = np.exp(h[n])
    E[n] = np.abs(y1_exact - y1)

plt.loglog(h,E,'b.-'), plt.grid(True)
plt.title("Método de Euler, $y'=y,y(0)=1$")
plt.xlabel("$h$"), plt.ylabel("Error de truncamiento local")
plt.show()

El gráfico logarítmico tiene pendiente 2 como se esperaba a partir de la fórmula de error y verificamos que el método de Euler es de orden 1.


---
### Ejercicio 3

Busque mostrar que RK4 es de orden 4

In [ ]:
# # # COMPLETAR

In [ ]:
# # # SOLUCION
f = lambda y,t: y
y0 = 1;
h = [0.1,0.05,0.01,0.005]
E = np.zeros(len(h))
for n in range(0,len(h)):
    tmax = 10
    t_grid = np.arange(0, tmax, step=h[n])
    y = np.zeros(len(t_grid))
    y[0] = y0
    for ix, tt in enumerate(t_grid[:-1]):
        y[ix+1] = rk4(f, y[ix], tt, h[n])
    y1 = y[1]
    y1_exact = np.exp(h[n])
    E[n] = np.abs(y1_exact - y1)
plt.loglog(h,E,'b.-'), plt.grid(True)
plt.title("Método de Euler, $y'=y,y(0)=1$")
plt.xlabel("$h$"), plt.ylabel("Error de truncamiento local")
plt.show()

<a name="scipy"></a>
## III. Funciones integradas en Scipy

Hay diversos métodos, más o menos precisos y que funcionan para diversos sistemas. Scipy trae un integrador bastante usado llamado [solve_ivp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.solve_ivp.html) (no teman a la documentación).

### **Ejemplo**

La manera de aplicarlo es bastante directa respecto a cómo veníamos. Vamos a ver como ejemplo el péndulo simple.

In [ ]:
# Parameters of the pendulum
g = 9.81  # Gravity (m/s^2)
l = 1.0   # Length of the pendulum (m)

# Function defining the ODEs
def pendulum(t, y):
    theta, omega = y
    dtheta_dt = omega
    domega_dt = -(g / l) * np.sin(theta)
    return [dtheta_dt, domega_dt]

In [ ]:
# Initial conditions: [initial angle, initial angular velocity]
theta0 = np.pi / 6  # 30 degrees in radians
omega0 = 0.0        # No initial velocity
y0 = [theta0, omega0]

# Time span for the simulation
t_span = (0, 10)  # 10 seconds
t_eval = np.linspace(t_span[0], t_span[1], 1000)

# Solve the ODE
solution = solve_ivp(pendulum, t_span, y0, t_eval=t_eval)

In [ ]:
# Extract results
time = solution.t
theta = solution.y[0]

# Plot the results
plt.figure(figsize=(8, 4))
plt.plot(time, theta)
plt.title("Pendulum Motion")
plt.xlabel("Time (s)")
plt.ylabel("Angle (rad)")
plt.grid()
plt.show()